In [1]:
#@title Environment & GPU
import torch, sys, os, platform, subprocess, textwrap
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU")


Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
#@title Install dependencies
!pip -q install --upgrade pip
!pip -q install torch torchvision albumentations opencv-python tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.7 MB/s eta 0:00:00


In [3]:
#@title Mount Google Drive (optional but recommended)
from google.colab import drive
drive.mount('/content/drive')

# Choose where to store data & runs in Drive:
DATA_DIR = "/content/drive/MyDrive/breast_ultrasound_data"  # change if you like
RUNS_DIR = "/content/drive/MyDrive/attnunet_busi_runs"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)


Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/breast_ultrasound_data
RUNS_DIR: /content/drive/MyDrive/attnunet_busi_runs


In [5]:
!pip install kagglehub

import kagglehub
path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print("Path to dataset files:", path)


100%|██████████| 195M/195M [00:11<00:00, 18.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/aryashah2k/breast-ultrasound-images-dataset/versions/1


In [6]:
#@title Medium Attention U-Net training for BUSI (80/10/10 split)

import os, random
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------------------------
# Repro
# ---------------------------
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ---------------------------
# Dataset
# ---------------------------
class BUSISegDataset(Dataset):
    """
    BUSI (Kaggle) loader with robust path handling.
    Uses benign+malignant (masked). 'normal' optional (zero masks).
    """
    def __init__(self, root: str, image_size: int = 512,
                 include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        candidate = self.root / "Dataset_BUSI_with_GT"
        base = candidate if candidate.exists() else self.root

        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")

        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            class_dir = base / c
            if not class_dir.exists(): continue
            for p in sorted(class_dir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    # match possible masks
                    stem = p.stem
                    candidates = []
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2"]:
                            candidates.append(p.with_name(stem + suf + ext))
                    exists = [q for q in candidates if q.exists()]
                    if len(exists) == 0:
                        continue
                    self.samples.append((str(p), str(exists[0])))

        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {self.root}")

        self.image_size = image_size
        if train:
            self.tf = A.Compose([
                A.LongestMaxSize(max_size=image_size),
                A.PadIfNeeded(image_size, image_size,
                              border_mode=cv2.BORDER_CONSTANT, border_value=0),
                A.HorizontalFlip(p=0.5),
                A.Affine(scale=(0.9,1.1), translate_percent=(0,0.05),
                         rotate=(-15,15), cval=0, mode=cv2.BORDER_CONSTANT, p=0.5),
                A.RandomBrightnessContrast(0.15,0.15,p=0.35),
                A.GaussNoise(var_limit=(5.0,15.0), p=0.25),
                A.GaussianBlur(blur_limit=(3,5), p=0.2),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2(),
            ])
        else:
            self.tf = A.Compose([
                A.LongestMaxSize(max_size=image_size),
                A.PadIfNeeded(image_size, image_size,
                              border_mode=cv2.BORDER_CONSTANT, border_value=0),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2(),
            ])

    def __len__(self): return len(self.samples)

    def _read_merge_masks(self, img_path: Path):
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
        stem = img_path.stem
        merged = None
        for ext in exts:
            for suf in ["_mask", "_mask_1", "_mask_2"]:
                p = img_path.with_name(stem + suf + ext)
                if p.exists():
                    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                    if m is None: continue
                    m = (m > 0).astype(np.uint8)
                    merged = m if merged is None else np.maximum(merged, m)
        return merged

    def __getitem__(self, idx: int):
        img_path, mask_path = self.samples[idx]
        img_path = Path(img_path)
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read {img_path}")

        if mask_path is None:
            mask = np.zeros_like(img, dtype=np.uint8)
        else:
            merged = self._read_merge_masks(img_path)
            if merged is not None:
                mask = merged
            else:
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: raise RuntimeError(f"Failed to read mask {mask_path}")
                mask = (m > 0).astype(np.uint8)

        aug = self.tf(image=img, mask=mask)
        x = aug["image"].float()
        x = x[:1] if x.ndim == 3 else x.unsqueeze(0)
        y = aug["mask"].unsqueeze(0).float()
        return x, y, str(img_path)

# ---------------------------
# Model: Attention U-Net (medium)
# ---------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UpConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
    def forward(self, x): return self.up(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True),
                                 nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True),
                                 nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),
                                 nn.BatchNorm2d(1),
                                 nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.relu(self.W_g(g) + self.W_x(x))
        psi = self.psi(psi)
        return x * psi

class AttnUNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, widths=(48,96,192,384,768)):
        super().__init__()
        c1,c2,c3,c4,c5 = widths
        self.enc1 = ConvBlock(in_ch, c1); self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(c1, c2);   self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(c2, c3);   self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(c3, c4);   self.pool4 = nn.MaxPool2d(2)
        self.center = ConvBlock(c4, c5)
        self.up4 = UpConv(c5, c4); self.att4 = AttentionGate(c4, c4, c4//2); self.dec4 = ConvBlock(c5, c4)
        self.up3 = UpConv(c4, c3); self.att3 = AttentionGate(c3, c3, c3//2); self.dec3 = ConvBlock(c4, c3)
        self.up2 = UpConv(c3, c2); self.att2 = AttentionGate(c2, c2, c2//2); self.dec2 = ConvBlock(c3, c2)
        self.up1 = UpConv(c2, c1); self.att1 = AttentionGate(c1, c1, c1//2); self.dec1 = ConvBlock(c2, c1)
        self.outc = nn.Conv2d(c1, out_ch, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2)); e4 = self.enc4(self.pool3(e3))
        c  = self.center(self.pool4(e4))
        d4 = self.up4(c); s4 = self.att4(d4, e4); d4 = self.dec4(torch.cat([s4, d4], 1))
        d3 = self.up3(d4); s3 = self.att3(d3, e3); d3 = self.dec3(torch.cat([s3, d3], 1))
        d2 = self.up2(d3); s2 = self.att2(d2, e2); d2 = self.dec2(torch.cat([s2, d2], 1))
        d1 = self.up1(d2); s1 = self.att1(d1, e1); d1 = self.dec1(torch.cat([s1, d1], 1))
        return self.outc(d1)

# ---------------------------
# Losses & metrics
# ---------------------------
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_w=0.5): super().__init__(); self.bce=nn.BCEWithLogitsLoss(); self.dice=DiceLoss(); self.w=bce_w
    def forward(self, logits, targets): return self.w*self.bce(logits, targets) + (1-self.w)*self.dice(logits, targets)

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum((2,3)) + eps
    den = probs.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs>thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds+targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs>thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    precision = (tp+eps)/(tp+fp+eps)
    recall    = (tp+eps)/(tp+fn+eps)
    f1 = 2*precision*recall/(precision+recall+eps)
    return f1.mean().item()

def run_eval(model, loader, device, amp=False):
    model.eval(); total=0; sum_dice=sum_iou=sum_f1=0.0
    with torch.no_grad():
        for x,y,_ in loader:
            x,y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); probs = torch.sigmoid(logits)
            sum_dice += dice_from_logits(logits, y) * x.size(0)
            sum_iou  += iou_from_probs(probs, y)     * x.size(0)
            sum_f1   += f1_from_probs(probs, y)      * x.size(0)
            total    += x.size(0)
    return sum_dice/total, sum_iou/total, sum_f1/total

def train_colab(data_root, outdir, include_normal=False, img_size=512, batch_size=6,
                epochs=40, lr=3e-4, workers=2, amp=True, seed=42):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    root = Path(data_root)
    candidate = root / "Dataset_BUSI_with_GT"
    data_base = candidate if candidate.exists() else root

    full_ds = BUSISegDataset(str(data_base), image_size=img_size,
                             include_normal=include_normal, train=False)
    n_total = len(full_ds)
    n_train = int(0.8*n_total); n_val = int(0.1*n_total); n_test = n_total - n_train - n_val
    print(f"Total={n_total} -> train={n_train}, val={n_val}, test={n_test}")

    g = torch.Generator().manual_seed(seed)
    train_idxs, val_idxs, test_idxs = random_split(full_ds, [n_train, n_val, n_test], generator=g)

    def subset_files(sub):
        return [full_ds.samples[i] for i in sub.indices]

    class BUSISubset(Dataset):
        def __init__(self, filelist, image_size, train_flag):
            self._files = filelist
            self.loader = BUSISegDataset(str(data_base), image_size=image_size,
                                         include_normal=include_normal, train=train_flag)
            self.loader.samples = self._files
        def __len__(self): return len(self._files)
        def __getitem__(self, i): return self.loader[i]

    train_ds = BUSISubset(subset_files(train_idxs), img_size, True)
    val_ds   = BUSISubset(subset_files(val_idxs),   img_size, False)
    test_ds  = BUSISubset(subset_files(test_idxs),  img_size, False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

    model = AttnUNet(in_ch=1, out_ch=1, widths=(48,96,192,384,768)).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = BCEDiceLoss(0.5)
    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.type=="cuda"))

    os.makedirs(outdir, exist_ok=True); best = -1.0

    for epoch in range(1, epochs+1):
        model.train(); run_loss=0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [train]")
        for x,y,_ in pbar:
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            run_loss += loss.item()*x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()
        tr_loss = run_loss/len(train_ds)

        val_dice, val_iou, val_f1 = run_eval(model, val_loader, device, amp=amp)
        print(f"Epoch {epoch}: train_loss={tr_loss:.4f} | val_dice={val_dice:.4f} | val_iou={val_iou:.4f} | val_f1={val_f1:.4f}")

        if val_dice > best:
            best = val_dice
            torch.save({"model": model.state_dict(),
                        "img_size": img_size, "widths": (48,96,192,384,768),
                        "epoch": epoch, "val_dice": val_dice}, os.path.join(outdir, "best.pt"))
            print(f"  ✓ Saved new best (val_dice={val_dice:.4f}) → {os.path.join(outdir, 'best.pt')}")

    # Test with best
    print("\nEvaluating best on TEST …")
    ckpt = torch.load(os.path.join(outdir, "best.pt"), map_location=device)
    model.load_state_dict(ckpt["model"])
    td, ti, tf1 = run_eval(model, test_loader, device, amp=amp)
    print(f"TEST → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}")



In [13]:
import kagglehub

# Download & get absolute path
DATA_DIR = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print("DATA_DIR =", DATA_DIR)

# Verify what's inside
!ls -R "$DATA_DIR" | head -n 40


DATA_DIR = /kaggle/input/breast-ultrasound-images-dataset
/kaggle/input/breast-ultrasound-images-dataset:
Dataset_BUSI_with_GT

/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT:
benign
malignant
normal

/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT/benign:
benign (100)_mask_1.png
benign (100)_mask.png
benign (100).png
benign (101)_mask.png
benign (101).png
benign (102)_mask.png
benign (102).png
benign (103)_mask.png
benign (103).png
benign (104)_mask.png
benign (104).png
benign (105)_mask.png
benign (105).png
benign (106)_mask.png
benign (106).png
benign (107)_mask.png
benign (107).png
benign (108)_mask.png
benign (108).png
benign (109)_mask.png
benign (109).png
benign (10)_mask.png
benign (10).png
benign (110)_mask.png
benign (110).png
benign (111)_mask.png
benign (111).png
benign (112)_mask.png
benign (112).png
benign (113)_mask.png
benign (113).png


In [14]:
#@title Kick off training
# If you mounted Drive and used the default DATA_DIR above, BUSI lives at:
BUSI_ROOT =DATA_DIR

train_colab(
    data_root=BUSI_ROOT,     # points to folder that contains "Dataset_BUSI_with_GT"
    outdir=RUNS_DIR,         # where best.pt will be saved
    include_normal=False,    # set True to add normal (zero masks)
    img_size=512,
    batch_size=6,            # tweak if you hit OOM; try 4
    epochs=40,
    lr=3e-4,
    workers=2,               # 2 is safe in Colab
    amp=True,                # mixed precision for speed
    seed=42
)


Device: cuda


/tmp/ipython-input-3356669818.py:85: UserWarning: Argument(s) 'border_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(image_size, image_size,
/tmp/ipython-input-3356669818.py:71: UserWarning: Argument(s) 'border_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(image_size, image_size,
/tmp/ipython-input-3356669818.py:74: UserWarning: Argument(s) 'cval, mode' are not valid for transform Affine
  A.Affine(scale=(0.9,1.1), translate_percent=(0,0.05),
/tmp/ipython-input-3356669818.py:77: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0,15.0), p=0.25),


Total=647 -> train=517, val=64, test=66
Params: 17.66M


/tmp/ipython-input-3356669818.py:290: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.type=="cuda"))
Epoch 1/40 [train]: 100%|██████████| 87/87 [01:24<00:00,  1.03it/s, loss=0.6215]


Epoch 1: train_loss=0.6381 | val_dice=0.1048 | val_iou=0.0000 | val_f1=0.0000
  ✓ Saved new best (val_dice=0.1048) → /content/drive/MyDrive/attnunet_busi_runs/best.pt


...

Epoch 38: train_loss=0.2609 | val_dice=0.5714 | val_iou=0.5137 | val_f1=0.6050


Epoch 39/40 [train]: 100%|██████████| 87/87 [00:09<00:00,  9.45it/s, loss=0.3044]


Epoch 39: train_loss=0.2617 | val_dice=0.5734 | val_iou=0.5144 | val_f1=0.6063


Epoch 40/40 [train]: 100%|██████████| 87/87 [00:09<00:00,  9.43it/s, loss=0.4167]


Epoch 40: train_loss=0.2695 | val_dice=0.5614 | val_iou=0.5133 | val_f1=0.6062

Evaluating best on TEST …
TEST → Dice=0.6084 | IoU=0.5553 | F1=0.6680


In [4]:
# %% [colab] BUSI Segmentation — UNet(ResNet34 pretrained), light augs, 80/10/10, early stop

!pip install segmentation_models_pytorch
# ====== CONFIG ======
DATA_DIR = None                 # None -> download via kagglehub; or set your dataset path string
RUNS_DIR = "/content/runs_busi_resnet34"
INCLUDE_NORMAL = False          # include "normal" images as zero masks
IMG_SIZE = 512
BATCH_SIZE = 8                  # ResNet34 encoder + 512^2 fits on T4 with bs~8; lower if OOM
EPOCHS = 80
LR = 1e-4                       # smaller LR for stability with pretrained encoder
WORKERS = 2
AMP = True
SEED = 42

# ====== SETUP ======
import os, random, warnings
from pathlib import Path
from typing import List, Tuple, Optional

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

!pip -q install segmentation-models-pytorch==0.3.3 timm==0.9.12 albumentations==1.4.8 opencv-python==4.9.0.80
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

# ====== DATA ACQUISITION (kagglehub optional) ======
def ensure_busi_data(data_dir: Optional[str]) -> str:
    if data_dir is None:
        print("DATA_DIR is None -> downloading with kagglehub…")
        try:
            import kagglehub
        except Exception as e:
            raise SystemExit("Please add: !pip -q install kagglehub") from e
        path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
        print("kagglehub path:", path)
        return str(path)
    else:
        p = Path(data_dir).expanduser().resolve()
        if not p.exists():
            raise SystemExit(f"DATA_DIR does not exist: {p}")
        return str(p)

def auto_find_busi_root(base_dir: str) -> str:
    base = Path(base_dir)
    cands = list(base.rglob("Dataset_BUSI_with_GT")) + list(base.rglob("Breast Ultrasound Images Dataset"))
    for p in cands:
        if (p/"benign").exists() and (p/"malignant").exists():
            print("Found dataset folder:", p)
            return str(p.parent)
    for p in base.rglob("benign"):
        if (p.parent/"malignant").exists():
            print("Found class folders under:", p.parent)
            return str(p.parent)
    return ""

DATA_DIR = ensure_busi_data(DATA_DIR)
BUSI_ROOT = auto_find_busi_root(DATA_DIR)
if not BUSI_ROOT:
    raise SystemExit("Could not locate BUSI. Inspect your DATA_DIR.")
print("Using BUSI_ROOT:", BUSI_ROOT)

# ====== DATASET ======
def _make_tf(image_size: int, train: bool):
    # Light, label-safe augs only
    if train:
        return A.Compose([
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=12,
                               border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
            A.RandomBrightnessContrast(0.10, 0.10, p=0.3),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])

class BUSISegDataset(Dataset):
    """
    BUSI loader. Accepts either:
      ROOT/Dataset_BUSI_with_GT/{benign,malignant,normal}
      or ROOT/{benign,malignant,normal}
    Uses benign+malignant by default (they have masks).
    """
    def __init__(self, root: str, image_size: int = 512, include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        candidate = self.root / "Dataset_BUSI_with_GT"
        base = candidate if candidate.exists() else self.root

        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")

        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            cdir = base / c
            if not cdir.exists(): continue
            for p in sorted(cdir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    stem = p.stem
                    mask_paths = []
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2"]:
                            q = p.with_name(stem + suf + ext)
                            if q.exists(): mask_paths.append(q)
                    if len(mask_paths) == 0:
                        continue
                    self.samples.append((str(p), str(mask_paths[0])))

        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {self.root}")

        self.tf = _make_tf(image_size, train)

    def __len__(self): return len(self.samples)

    def _read_merge_masks(self, img_path: Path) -> np.ndarray:
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
        stem = img_path.stem
        merged = None
        for ext in exts:
            for suf in ["_mask", "_mask_1", "_mask_2"]:
                q = img_path.with_name(stem + suf + ext)
                if q.exists():
                    m = cv2.imread(str(q), cv2.IMREAD_GRAYSCALE)
                    if m is None: continue
                    m = (m > 0).astype(np.uint8)
                    merged = m if merged is None else np.maximum(merged, m)
        return merged

    def __getitem__(self, i: int):
        img_path, mask_path = self.samples[i]
        img_path = Path(img_path)
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read image {img_path}")

        if mask_path is None:
            mask = np.zeros_like(img, dtype=np.uint8)
        else:
            merged = self._read_merge_masks(img_path)
            if merged is not None:
                mask = merged
            else:
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: raise RuntimeError(f"Failed to read mask {mask_path}")
                mask = (m > 0).astype(np.uint8)

        aug = self.tf(image=img, mask=mask)
        x = aug["image"].float()
        # ensure 1-channel tensor
        x = x[:1] if x.ndim == 3 else x.unsqueeze(0)
        y = aug["mask"].unsqueeze(0).float()
        return x, y, str(img_path)

# ====== MODEL: UNet (ResNet34 encoder, pretrained), 1 input ch ======
# smp allows in_channels=1 with pretrained weights by adapting first conv automatically.
def build_model():
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1,
        activation=None
    )
    return model

# ====== LOSSES & METRICS ======
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

class BCEDice(nn.Module):
    def __init__(self, w_bce=0.4): super().__init__(); self.bce=nn.BCEWithLogitsLoss(); self.dice=DiceLoss(); self.w=w_bce
    def forward(self, logits, targets): return self.w*self.bce(logits, targets) + (1-self.w)*self.dice(logits, targets)

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum((2,3)) + eps
    den = probs.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds + targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    precision = (tp+eps)/(tp+fp+eps)
    recall    = (tp+eps)/(tp+fn+eps)
    f1 = 2*precision*recall/(precision+recall+eps)
    return f1.mean().item()

def run_eval(model, loader, device, amp=False, thr=0.5):
    model.eval(); total=0; d=iou=f1=0.0
    with torch.no_grad():
        for x,y,_ in loader:
            x,y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); probs = torch.sigmoid(logits)
            d  += dice_from_logits(logits, y) * x.size(0)
            iou+= iou_from_probs(probs, y, thr=thr) * x.size(0)
            f1 += f1_from_probs(probs, y, thr=thr) * x.size(0)
            total += x.size(0)
    return d/total, iou/total, f1/total

# ====== TRAIN ======
def train_main(busi_root: str, runs_dir: str, include_normal=False, img_size=512,
               batch_size=8, epochs=80, lr=1e-4, workers=2, amp=True, seed=42):
    set_seed(seed)

    root = Path(busi_root)
    candidate = root / "Dataset_BUSI_with_GT"
    data_base = candidate if candidate.exists() else root
    print("Resolved data base:", data_base)

    full_ds = BUSISegDataset(str(data_base), image_size=img_size, include_normal=include_normal, train=False)
    n_total = len(full_ds)
    n_tr = int(0.8*n_total); n_va = int(0.1*n_total); n_te = n_total - n_tr - n_va
    print(f"Total={n_total} -> train={n_tr}, val={n_va}, test={n_te}")

    g = torch.Generator().manual_seed(seed)
    tr_idx, va_idx, te_idx = random_split(full_ds, [n_tr, n_va, n_te], generator=g)

    def subfiles(sub): return [full_ds.samples[i] for i in sub.indices]

    class BUSISubset(Dataset):
        def __init__(self, filelist, image_size, train_flag):
            self.files = filelist
            self.loader = BUSISegDataset(str(data_base), image_size=image_size,
                                         include_normal=include_normal, train=train_flag)
            self.loader.samples = self.files
        def __len__(self): return len(self.files)
        def __getitem__(self, i): return self.loader[i]

    train_ds = BUSISubset(subfiles(tr_idx), img_size, True)
    val_ds   = BUSISubset(subfiles(va_idx), img_size, False)
    test_ds  = BUSISubset(subfiles(te_idx), img_size, False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

    model = build_model().to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = BCEDice(w_bce=0.4)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(amp and device.type=="cuda"))

    os.makedirs(runs_dir, exist_ok=True)
    best_dice, best_path = -1.0, os.path.join(runs_dir, "best.pt")
    patience, bad_epochs = 10, 0

    for ep in range(1, epochs+1):
        model.train(); run_loss=0.0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs} [train]")
        for x,y,_ in pbar:
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            run_loss += loss.item()*x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        scheduler.step()
        tr_loss = run_loss/len(train_ds)

        vd, vi, vf = run_eval(model, val_loader, device, amp=amp, thr=0.5)
        print(f"Epoch {ep}: train_loss={tr_loss:.4f} | val_dice={vd:.4f} | val_iou={vi:.4f} | val_f1={vf:.4f}")

        if vd > best_dice:
            best_dice = vd; bad_epochs = 0
            torch.save({"model": model.state_dict(),
                        "img_size": img_size,
                        "encoder": "resnet34",
                        "epoch": ep,
                        "val_dice": vd}, best_path)
            print(f"  ✓ Saved new best (Dice={vd:.4f}) → {best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {ep} (no val Dice improvement for {patience} epochs)")
                break

    # Reload best, sweep threshold on VAL, evaluate TEST
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])

    def sweep_threshold(loader, steps=11):
        best_thr, best_iou = 0.5, -1.0
        for k in range(steps):
            thr = 0.3 + k*(0.4/(steps-1))  # 0.3..0.7
            d,i,f = run_eval(model, loader, device, amp=amp, thr=thr)
            if i > best_iou:
                best_iou, best_thr = i, thr
        return best_thr, best_iou

    best_thr, val_iou_at_thr = sweep_threshold(val_loader, steps=11)
    print(f"Chosen threshold from VAL: {best_thr:.3f} (val IoU={val_iou_at_thr:.4f})")

    td, ti, tf1 = run_eval(model, test_loader, device, amp=amp, thr=best_thr)
    print(f"TEST → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}  @thr={best_thr:.3f}")

# ====== GO ======
train_main(BUSI_ROOT, RUNS_DIR,
           include_normal=INCLUDE_NORMAL,
           img_size=IMG_SIZE,
           batch_size=BATCH_SIZE,
           epochs=EPOCHS,
           lr=LR,
           workers=WORKERS,
           amp=AMP,
           seed=SEED)


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install segmentation-models-pytorch==0.3.3 and timm==0.9.12 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
Device: cuda
DATA_DIR is None -> downloading with kagglehub…
kagglehub path: /kaggle/input/breast-ultrasound-images-dataset
Found dataset folder: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Using BUSI_ROOT: /kaggle/input/breast-ultrasound-images-dataset
Resolved data base: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Total=647 -> train=517, val=64, test=66
Params: 24.43M


Epoch 1/80 [train]: 100%|██████████| 65/65 [00:08<00:00,  7.47it/s, loss=0.6703]


Epoch 1: train_loss=0.7110 | val_dice=0.0000 | val_iou=0.3079 | val_f1=0.4348
  ✓ Saved new best (Dice=0.0000) → /content/runs_busi_resnet34/best.pt


Epoch 2/80 [train]: 100%|██████████| 65/65 [00:08<00:00,  7.46it/s, loss=0.5488]


Epoch 2: train_loss=0.5916 | val_dice=0.0186 | val_iou=0.5057 | val_f1=0.6271
  ✓ Saved new best (Dice=0.0186) → /content/runs_busi_resnet34/best.pt


...

Epoch 37: train_loss=0.0729 | val_dice=0.6922 | val_iou=0.6583 | val_f1=0.7281


Epoch 38/80 [train]: 100%|██████████| 65/65 [00:08<00:00,  7.50it/s, loss=0.0707]


Epoch 38: train_loss=0.0756 | val_dice=0.6995 | val_iou=0.6484 | val_f1=0.7204


Epoch 39/80 [train]: 100%|██████████| 65/65 [00:08<00:00,  7.48it/s, loss=0.1457]


Epoch 39: train_loss=0.0703 | val_dice=0.6939 | val_iou=0.6481 | val_f1=0.7183
Early stopping at epoch 39 (no val Dice improvement for 10 epochs)
Chosen threshold from VAL: 0.300 (val IoU=0.6598)
TEST → Dice=0.7149 | IoU=0.6863 | F1=0.7678  @thr=0.300


In [5]:
# %% [colab] BUSI Segmentation — UNet++(EffB4 pretrained), loss cocktail, multi-scale + TTA, no postproc
!pip install segmentation_models_pytorch

# ========== CONFIG ==========
DATA_DIR = None                 # None -> auto-download with kagglehub; or set your dataset path string
RUNS_DIR = "/content/runs_busi_unetpp_effb4"
INCLUDE_NORMAL = False          # include "normal" images as zero masks (usually False for BUSI seg)
IMG_SIZE = 512                  # base train/eval image size
BATCH_SIZE = 8
EPOCHS = 120
LR = 1e-4
WORKERS = 2
AMP = True
SEED = 42
AUG_CLAHE = True                # set False if you don't want CLAHE

# ========== SETUP ==========
import os, random, warnings, math
from pathlib import Path
from typing import List, Tuple, Optional

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from tqdm import tqdm

# core deps (smp already present in your environment)
!pip -q install --no-input albumentations opencv-python
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

# ========== DATA ACQUISITION (kagglehub optional) ==========
def ensure_busi_data(data_dir: Optional[str]) -> str:
    if data_dir is None:
        print("DATA_DIR is None -> downloading with kagglehub…")
        try:
            import kagglehub
        except Exception as e:
            raise SystemExit("Add: !pip -q install kagglehub") from e
        path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
        print("kagglehub path:", path)
        return str(path)
    else:
        p = Path(data_dir).expanduser().resolve()
        if not p.exists():
            raise SystemExit(f"DATA_DIR does not exist: {p}")
        return str(p)

def auto_find_busi_root(base_dir: str) -> str:
    base = Path(base_dir)
    cands = list(base.rglob("Dataset_BUSI_with_GT")) + list(base.rglob("Breast Ultrasound Images Dataset"))
    for p in cands:
        if (p/"benign").exists() and (p/"malignant").exists():
            print("Found dataset folder:", p)
            return str(p.parent)
    for p in base.rglob("benign"):
        if (p.parent/"malignant").exists():
            print("Found class folders under:", p.parent)
            return str(p.parent)
    return ""

DATA_DIR = ensure_busi_data(DATA_DIR)
BUSI_ROOT = auto_find_busi_root(DATA_DIR)
if not BUSI_ROOT:
    raise SystemExit("Could not locate BUSI inside DATA_DIR.")
print("Using BUSI_ROOT:", BUSI_ROOT)

# ========== DATASET ==========
def _make_tf(image_size: int, train: bool):
    # Light, ultrasound-friendly augs + multi-scale jitter
    if train:
        tf = [
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.HorizontalFlip(p=0.5),
            # mild multi-scale + small geometry jitter
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=12,
                               border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6),
            A.RandomBrightnessContrast(0.10, 0.10, p=0.3),
        ]
        if AUG_CLAHE:
            tf.insert(0, A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.2))
        tf += [
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ]
        return A.Compose(tf)
    else:
        return A.Compose([
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])

class BUSISegDataset(Dataset):
    """
    BUSI loader. Accepts either:
      ROOT/Dataset_BUSI_with_GT/{benign,malignant,normal}
      or ROOT/{benign,malignant,normal}
    Uses benign+malignant by default (they have masks).
    """
    def __init__(self, root: str, image_size: int = 512, include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        candidate = self.root / "Dataset_BUSI_with_GT"
        base = candidate if candidate.exists() else self.root

        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")

        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            cdir = base / c
            if not cdir.exists(): continue
            for p in sorted(cdir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    stem = p.stem
                    mask_paths = []
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2"]:
                            q = p.with_name(stem + suf + ext)
                            if q.exists(): mask_paths.append(q)
                    if len(mask_paths) == 0:
                        continue
                    self.samples.append((str(p), str(mask_paths[0])))

        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {self.root}")

        self.tf = _make_tf(image_size, train)
        self.train_flag = train
        self.image_size = image_size

        # Precompute tiny-mask flags for sampler (fast scan)
        self.mask_area_frac = None
        if train:
            areas = []
            for img_path, mask_path in self.samples:
                if mask_path is None:
                    areas.append(0.0)
                    continue
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: areas.append(0.0); continue
                a = float((m > 0).sum()) / (m.shape[0] * m.shape[1])
                areas.append(a)
            self.mask_area_frac = np.array(areas, dtype=np.float32)

    def __len__(self): return len(self.samples)

    def _read_merge_masks(self, img_path: Path) -> np.ndarray:
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
        stem = img_path.stem
        merged = None
        for ext in exts:
            for suf in ["_mask", "_mask_1", "_mask_2"]:
                q = img_path.with_name(stem + suf + ext)
                if q.exists():
                    m = cv2.imread(str(q), cv2.IMREAD_GRAYSCALE)
                    if m is None: continue
                    m = (m > 0).astype(np.uint8)
                    merged = m if merged is None else np.maximum(merged, m)
        return merged

    def __getitem__(self, i: int):
        img_path, mask_path = self.samples[i]
        img_path = Path(img_path)
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read image {img_path}")

        if mask_path is None:
            mask = np.zeros_like(img, dtype=np.uint8)
        else:
            merged = self._read_merge_masks(img_path)
            if merged is not None:
                mask = merged
            else:
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: raise RuntimeError(f"Failed to read mask {mask_path}")
                mask = (m > 0).astype(np.uint8)

        aug = self.tf(image=img, mask=mask)
        x = aug["image"].float()
        x = x[:1] if x.ndim == 3 else x.unsqueeze(0)  # 1-channel
        y = aug["mask"].unsqueeze(0).float()
        return x, y, str(img_path)

# ========== MODEL ==========
def build_model():
    model = smp.UnetPlusPlus(
        encoder_name="timm-efficientnet-b4",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1,
        activation=None
    )
    return model

# ========== LOSSES & METRICS ==========
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, eps=1e-7):
        super().__init__()
        self.a, self.b, self.g, self.eps = alpha, beta, gamma, eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        tp = (p*targets).sum((2,3))
        fp = (p*(1-targets)).sum((2,3))
        fn = ((1-p)*targets).sum((2,3))
        tversky = (tp + self.eps) / (tp + self.a*fp + self.b*fn + self.eps)
        return torch.pow(1 - tversky, self.g).mean()

class BCE_Dice_FT(nn.Module):
    def __init__(self, wbce=0.4, wdice=0.4, wft=0.2):
        super().__init__()
        self.wb, self.wd, self.wt = wbce, wdice, wft
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.ft   = FocalTverskyLoss(0.7,0.3,0.75)
    def forward(self, logits, targets):
        return self.wb*self.bce(logits, targets) + self.wd*self.dice(logits, targets) + self.wt*self.ft(logits, targets)

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum((2,3)) + eps
    den = probs.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds + targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    prec = (tp+eps)/(tp+fp+eps)
    rec  = (tp+eps)/(tp+fn+eps)
    return (2*prec*rec/(prec+rec+eps)).mean().item()

# ========== EVAL (with optional TTA) ==========
def tta_predict(model, x, amp=False, scales=(1.0, 1.25), hflip=True):
    """
    x: [B,1,H,W] normalized
    returns mean probability over TTA variants
    """
    outs = []
    B, C, H, W = x.shape
    for s in scales:
        if s == 1.0:
            xs = x
        else:
            Hs, Ws = int(round(H*s)), int(round(W*s))
            xs = torch.nn.functional.interpolate(x, size=(Hs, Ws), mode="bilinear", align_corners=False)
        # original
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            lo = model(xs); po = torch.sigmoid(lo)
        if s != 1.0:
            po = torch.nn.functional.interpolate(po, size=(H, W), mode="bilinear", align_corners=False)
        outs.append(po)
        # horizontal flip
        if hflip:
            xh = torch.flip(xs, dims=[-1])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                lh = model(xh); ph = torch.sigmoid(lh)
            ph = torch.flip(ph, dims=[-1])
            if s != 1.0:
                ph = torch.nn.functional.interpolate(ph, size=(H, W), mode="bilinear", align_corners=False)
            outs.append(ph)
    return torch.stack(outs, dim=0).mean(dim=0)  # [B,1,H,W]

def run_eval(model, loader, device, amp=False, thr=0.5, use_tta=False):
    model.eval(); total=0; d=iou=f1=0.0
    with torch.no_grad():
        for x,y,_ in loader:
            x,y = x.to(device), y.to(device)
            if use_tta:
                probs = tta_predict(model, x, amp=amp)
                logits = torch.logit(torch.clamp(probs, 1e-6, 1-1e-6))
            else:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    logits = model(x); probs = torch.sigmoid(logits)
            d  += dice_from_logits(logits, y) * x.size(0)
            iou+= iou_from_probs(probs, y, thr=thr) * x.size(0)
            f1 += f1_from_probs(probs, y, thr=thr) * x.size(0)
            total += x.size(0)
    return d/total, iou/total, f1/total

# ========== TRAIN ==========
def build_train_sampler(train_ds: BUSISegDataset, tiny_quantile=0.30, tiny_boost=2.0):
    """
    Oversample images whose mask area fraction is in the bottom quantile (tiny lesions).
    """
    areas = train_ds.mask_area_frac
    if areas is None: return None  # fallback to default shuffling
    thr = np.quantile(areas, tiny_quantile)
    w = np.ones(len(areas), dtype=np.float32)
    w[areas <= thr] = tiny_boost
    sampler = WeightedRandomSampler(weights=torch.from_numpy(w), num_samples=len(w), replacement=True)
    print(f"Sampler: tiny<=Q{int(tiny_quantile*100)} boosted x{tiny_boost:.1f} (thr={thr:.6f})")
    return sampler

def train_main(busi_root: str, runs_dir: str, include_normal=False, img_size=512,
               batch_size=8, epochs=120, lr=1e-4, workers=2, amp=True, seed=42):
    set_seed(seed)

    root = Path(busi_root)
    candidate = root / "Dataset_BUSI_with_GT"
    data_base = candidate if candidate.exists() else root
    print("Resolved data base:", data_base)

    full_ds = BUSISegDataset(str(data_base), image_size=img_size, include_normal=include_normal, train=False)
    n_total = len(full_ds)
    n_tr = int(0.8*n_total); n_va = int(0.1*n_total); n_te = n_total - n_tr - n_va
    print(f"Total={n_total} -> train={n_tr}, val={n_va}, test={n_te}")

    g = torch.Generator().manual_seed(seed)
    tr_idx, va_idx, te_idx = random_split(full_ds, [n_tr, n_va, n_te], generator=g)

    def subfiles(sub): return [full_ds.samples[i] for i in sub.indices]

    class BUSISubset(Dataset):
        def __init__(self, filelist, image_size, train_flag):
            self.files = filelist
            self.loader = BUSISegDataset(str(data_base), image_size=image_size,
                                         include_normal=include_normal, train=train_flag)
            self.loader.samples = self.files
        def __len__(self): return len(self.files)
        def __getitem__(self, i): return self.loader[i]


    # Build three *separate* datasets (train has augs; val/test no augs)
    train_full = BUSISegDataset(str(data_base), image_size=img_size,
                                include_normal=include_normal, train=True)
    val_full   = BUSISegDataset(str(data_base), image_size=img_size,
                                include_normal=include_normal, train=False)
    test_full  = BUSISegDataset(str(data_base), image_size=img_size,
                                include_normal=include_normal, train=False)

    # Use Subset to avoid custom wrappers (no index mismatch)
    from torch.utils.data import Subset
    train_ds = Subset(train_full, tr_idx.indices)
    val_ds   = Subset(val_full,   va_idx.indices)
    test_ds  = Subset(test_full,  te_idx.indices)

    # ---- sampler for tiny lesions (build weights from the *subset* indices)
    sampler = None
    if train_full.mask_area_frac is not None:
        areas = train_full.mask_area_frac[np.array(tr_idx.indices)]
        thr = np.quantile(areas, 0.30)
        w = np.ones_like(areas, dtype=np.float32)
        w[areas <= thr] = 2.0
        from torch.utils.data import WeightedRandomSampler
        sampler = WeightedRandomSampler(weights=torch.from_numpy(w),
                                        num_samples=len(w), replacement=True)
        print(f"Sampler: tiny<=Q30 boosted x2.0 (thr={thr:.6f}) on train subset")

    # ---- DataLoaders (tip: if you still see dataloader hiccups, set WORKERS=0 temporarily)
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=(sampler is None), sampler=sampler,
                              num_workers=WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)
    model = build_model().to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = BCE_Dice_FT(0.4, 0.4, 0.2)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(amp and device.type=="cuda"))

    os.makedirs(runs_dir, exist_ok=True)
    best_dice, best_path = -1.0, os.path.join(runs_dir, "best.pt")
    patience, bad_epochs = 15, 0

    for ep in range(1, epochs+1):
        model.train(); run_loss=0.0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs} [train]")
        for x,y,_ in pbar:
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            scheduler.step(ep + (pbar.n / len(pbar)))  # warm restarts within-epoch step
            run_loss += loss.item()*x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        tr_loss = run_loss/len(train_ds)

        vd, vi, vf = run_eval(model, val_loader, device, amp=amp, thr=0.5, use_tta=False)  # selection at thr=0.5, no TTA
        print(f"Epoch {ep}: train_loss={tr_loss:.4f} | val_dice={vd:.4f} | val_iou={vi:.4f} | val_f1={vf:.4f}")

        if vd > best_dice:
            best_dice = vd; bad_epochs = 0
            torch.save({"model": model.state_dict(),
                        "img_size": img_size,
                        "encoder": "timm-efficientnet-b4",
                        "epoch": ep,
                        "val_dice": vd}, best_path)
            print(f"  ✓ Saved new best (Dice={vd:.4f}) → {best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {ep} (no val Dice improvement for {patience} epochs)")
                break

    # Reload best, sweep threshold on VAL (no TTA) and then evaluate TEST with TTA
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])

    def sweep_threshold(loader, steps=11, use_tta=False):
        best_thr, best_iou = 0.5, -1.0
        for k in range(steps):
            thr = 0.3 + k*(0.4/(steps-1))  # 0.3..0.7
            d,i,f = run_eval(model, loader, device, amp=amp, thr=thr, use_tta=use_tta)
            if i > best_iou:
                best_iou, best_thr = i, thr
        return best_thr, best_iou

    best_thr, val_iou_at_thr = sweep_threshold(val_loader, steps=11, use_tta=False)
    print(f"Chosen threshold from VAL (no TTA): {best_thr:.3f} (val IoU={val_iou_at_thr:.4f})")

    td, ti, tf1 = run_eval(model, test_loader, device, amp=amp, thr=best_thr, use_tta=True)
    print(f"TEST (with TTA) → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}  @thr={best_thr:.3f}")
    print("Best checkpoint:", best_path)

# ========== GO ==========
train_main(BUSI_ROOT, RUNS_DIR,
           include_normal=INCLUDE_NORMAL,
           img_size=IMG_SIZE,
           batch_size=BATCH_SIZE,
           epochs=EPOCHS,
           lr=LR,
           workers=WORKERS,
           amp=AMP,
           seed=SEED)


Device: cuda
DATA_DIR is None -> downloading with kagglehub…
kagglehub path: /kaggle/input/breast-ultrasound-images-dataset
Found dataset folder: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Using BUSI_ROOT: /kaggle/input/breast-ultrasound-images-dataset
Resolved data base: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Total=647 -> train=517, val=64, test=66
Sampler: tiny<=Q30 boosted x2.0 (thr=0.028477) on train subset
Params: 20.81M


Epoch 1/120 [train]: 100%|██████████| 65/65 [03:21<00:00,  3.11s/it, loss=0.7138]


Epoch 1: train_loss=0.8129 | val_dice=0.0000 | val_iou=0.3134 | val_f1=0.4358
  ✓ Saved new best (Dice=0.0000) → /content/runs_busi_unetpp_effb4/best.pt


Epoch 2/120 [train]: 100%|██████████| 65/65 [00:25<00:00,  2.51it/s, loss=0.6681]


Epoch 2: train_loss=0.6870 | val_dice=0.0000 | val_iou=0.4696 | val_f1=0.5858


...

Epoch 95: train_loss=0.0563 | val_dice=0.7338 | val_iou=0.6754 | val_f1=0.7506


Epoch 96/120 [train]: 100%|██████████| 65/65 [00:25<00:00,  2.52it/s, loss=0.0620]


Epoch 96: train_loss=0.0523 | val_dice=0.7324 | val_iou=0.6731 | val_f1=0.7491


Epoch 97/120 [train]: 100%|██████████| 65/65 [00:25<00:00,  2.52it/s, loss=0.0511]


Epoch 97: train_loss=0.0558 | val_dice=0.7327 | val_iou=0.6785 | val_f1=0.7494
Early stopping at epoch 97 (no val Dice improvement for 15 epochs)
Chosen threshold from VAL (no TTA): 0.300 (val IoU=0.6852)
TEST (with TTA) → Dice=0.7567 | IoU=0.7225 | F1=0.8085  @thr=0.300
Best checkpoint: /content/runs_busi_unetpp_effb4/best.pt


In [3]:
# %% [colab] BUSI Segmentation — BIG UNet++(EfficientNet-B7), heavy dropout, US-friendly augs, TTA
!pip install segmentation_models_pytorch
# =================== CONFIG ===================
DATA_DIR = None                 # None -> auto-download with kagglehub; or set to your mounted path
RUNS_DIR = "/content/runs_busi_unetpp_effb7_big"
INCLUDE_NORMAL = False          # include "normal" (no masks) as zero masks
IMG_SIZE = 512                  # reduce to 480/448 if CUDA OOM
BATCH_SIZE = 4                  # reduce to 3/2 if CUDA OOM
EPOCHS = 120
LR = 8e-5                       # slightly lower LR for a much larger net
WORKERS = 2                     # set 0 if DataLoader multiprocessing causes issues
AMP = True
SEED = 42
AUG_CLAHE = True                # subtle contrast enhancement
TTA_SCALES = (1.0, 1.25)        # TTA scales
TINY_Q = 0.30                   # oversample bottom-30% lesion area
TINY_BOOST = 2.0                # weight multiplier for tiny lesions

# =================== SETUP ===================
import os, random, warnings, math
from pathlib import Path
from typing import List, Tuple, Optional

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset, WeightedRandomSampler
from tqdm import tqdm

# deps
!pip -q install --no-input kagglehub albumentations opencv-python segmentation_models_pytorch
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

# =================== DATA ACQUISITION ===================
def ensure_busi_data(data_dir: Optional[str]) -> str:
    if data_dir is None:
        print("DATA_DIR is None -> downloading with kagglehub…")
        import kagglehub
        path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
        print("kagglehub path:", path)
        return str(path)
    else:
        p = Path(data_dir).expanduser().resolve()
        if not p.exists():
            raise SystemExit(f"DATA_DIR does not exist: {p}")
        return str(p)

def auto_find_busi_root(base_dir: str) -> str:
    base = Path(base_dir)
    # Known folder names
    cands = list(base.rglob("Dataset_BUSI_with_GT")) + list(base.rglob("Breast Ultrasound Images Dataset"))
    for p in cands:
        # Folder either *is* class root, or its parent is
        if (p/"benign").exists() and (p/"malignant").exists():
            print("Found dataset folder:", p)
            return str(p)
        if (p.parent/"benign").exists() and (p.parent/"malignant").exists():
            print("Found dataset folder:", p.parent)
            return str(p.parent)
    # fallback: find class dirs anywhere
    for p in base.rglob("benign"):
        if (p.parent/"malignant").exists():
            print("Found class folders under:", p.parent)
            return str(p.parent)
    return ""

DATA_DIR = ensure_busi_data(DATA_DIR)
BUSI_BASE = auto_find_busi_root(DATA_DIR)
if not BUSI_BASE:
    raise SystemExit("Could not locate BUSI inside DATA_DIR.")
print("Using BUSI_ROOT:", BUSI_BASE)

# resolve class root
candidate = Path(BUSI_BASE) / "Dataset_BUSI_with_GT"
DATA_BASE = candidate if candidate.exists() else Path(BUSI_BASE)
print("Resolved data base:", DATA_BASE)

# =================== DATASET ===================
def _make_tf(image_size: int, train: bool):
    if train:
        tf = [
            # keep grayscale look; CLAHE helps contrast a bit on US
            *( [A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.2)] if AUG_CLAHE else [] ),
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.HorizontalFlip(p=0.5),
            # mild geometry jitter; ShiftScaleRotate is cheap & safe for US
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=12,
                               border_mode=cv2.BORDER_CONSTANT, value=0, p=0.6),
            A.RandomBrightnessContrast(0.10, 0.10, p=0.3),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ]
        return A.Compose(tf)
    else:
        return A.Compose([
            A.LongestMaxSize(max_size=image_size),
            A.PadIfNeeded(image_size, image_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])

class BUSISegDataset(Dataset):
    """
    Accepts:
      DATA_BASE/{benign,malignant,normal} or DATA_BASE/Dataset_BUSI_with_GT/{...}
    Uses (benign + malignant) by default; optional "normal" as zero-mask.
    Merges multiple *_mask_* if present.
    """
    def __init__(self, root: str, image_size: int = 512, include_normal: bool = False, train: bool = False):
        self.root = Path(root)
        base = self.root if (self.root/"benign").exists() else (self.root/"Dataset_BUSI_with_GT")
        classes = ["benign", "malignant"] + (["normal"] if include_normal else [])
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")

        self.samples: List[Tuple[str, Optional[str]]] = []
        for c in classes:
            cdir = base / c
            if not cdir.exists(): continue
            for p in sorted(cdir.iterdir()):
                if not p.is_file(): continue
                if "_mask" in p.stem: continue
                if p.suffix not in exts: continue
                if c == "normal":
                    self.samples.append((str(p), None))
                else:
                    # accept _mask, _mask_1, _mask_2 …
                    stem = p.stem
                    found = False
                    for ext in exts:
                        for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                            q = p.with_name(stem + suf + ext)
                            if q.exists():
                                self.samples.append((str(p), str(q)))
                                found = True; break
                        if found: break
                    if not found:
                        # some files may have only image; skip
                        pass

        if len(self.samples) == 0:
            raise RuntimeError(f"No BUSI samples under {base}")

        self.tf = _make_tf(image_size, train)
        self.train_flag = train
        self.image_size = image_size

        # Precompute tiny-mask flags for sampler (only for train)
        self.mask_area_frac = None
        if train:
            areas = []
            for img_path, mask_path in self.samples:
                if mask_path is None:
                    areas.append(0.0); continue
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: areas.append(0.0); continue
                a = float((m > 0).sum()) / (m.shape[0] * m.shape[1])
                areas.append(a)
            self.mask_area_frac = np.array(areas, dtype=np.float32)

    def __len__(self): return len(self.samples)

    def _read_merge_masks(self, img_path: Path) -> np.ndarray:
        exts = (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG")
        stem = img_path.stem
        merged = None
        for ext in exts:
            for suf in ["_mask", "_mask_1", "_mask_2", "_mask_3"]:
                q = img_path.with_name(stem + suf + ext)
                if q.exists():
                    m = cv2.imread(str(q), cv2.IMREAD_GRAYSCALE)
                    if m is None: continue
                    m = (m > 0).astype(np.uint8)
                    merged = m if merged is None else np.maximum(merged, m)
        return merged

    def __getitem__(self, i: int):
        img_path, mask_path = self.samples[i]
        ip = Path(img_path)
        img = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
        if img is None: raise RuntimeError(f"Failed to read image {ip}")

        if mask_path is None:
            mask = np.zeros_like(img, dtype=np.uint8)
        else:
            merged = self._read_merge_masks(ip)
            if merged is not None:
                mask = merged
            else:
                m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if m is None: raise RuntimeError(f"Failed to read mask {mask_path}")
                mask = (m > 0).astype(np.uint8)

        aug = self.tf(image=img, mask=mask)
        x = aug["image"].float()
        x = x[:1] if x.ndim == 3 else x.unsqueeze(0)  # keep 1-channel
        y = aug["mask"].unsqueeze(0).float()
        return x, y, str(ip)

# =================== MODEL (BIG) ===================
from segmentation_models_pytorch.base import SegmentationHead

def build_model():
    """
    ~5x+ larger than EffB4 version:
      - Encoder: timm-efficientnet-b7 (pretrained)
      - Wider decoder: [512, 256, 128, 64, 32]
      - Segmentation head dropout: 0.50
      - 1 input channel, 1 output class
    """
    model = smp.UnetPlusPlus(
        encoder_name="timm-efficientnet-b7",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1,
        activation=None,
        decoder_channels=[512, 256, 128, 64, 32],
        decoder_use_batchnorm=True,
    )
    # inject strong dropout (0.50) in segmentation head
    old_head = model.segmentation_head
    head_in_ch = None
    if isinstance(old_head, nn.Sequential):
        for m in old_head.modules():
            if isinstance(m, nn.Conv2d):
                head_in_ch = m.in_channels
                break
    if head_in_ch is None:  # fallback
        head_in_ch = getattr(model.decoder, "out_channels", 256)

    model.segmentation_head = nn.Sequential(
        nn.Dropout2d(p=0.50, inplace=False),
        model.segmentation_head
    )
    return model

# =================== LOSSES & METRICS ===================
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7): super().__init__(); self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum((2,3)) + self.eps
        den = probs.sum((2,3)) + targets.sum((2,3)) + self.eps
        return 1 - (num/den).mean()

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3, gamma=0.75, eps=1e-7):
        super().__init__(); self.a, self.b, self.g, self.eps = alpha, beta, gamma, eps
    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        tp = (p*targets).sum((2,3))
        fp = (p*(1-targets)).sum((2,3))
        fn = ((1-p)*targets).sum((2,3))
        tversky = (tp + self.eps) / (tp + self.a*fp + self.b*fn + self.eps)
        return torch.pow(1 - tversky, self.g).mean()

class BCE_Dice_FT(nn.Module):
    def __init__(self, wbce=0.4, wdice=0.4, wft=0.2):
        super().__init__()
        self.wb, self.wd, self.wt = wbce, wdice, wft
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.ft   = FocalTverskyLoss(0.7,0.3,0.75)
    def forward(self, logits, targets):
        return self.wb*self.bce(logits, targets) + self.wd*self.dice(logits, targets) + self.wt*self.ft(logits, targets)

@torch.no_grad()
def dice_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2*(probs*targets).sum((2,3)) + eps
    den = probs.sum((2,3)) + targets.sum((2,3)) + eps
    return (num/den).mean().item()

@torch.no_grad()
def iou_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    inter = (preds*targets).sum((2,3))
    union = (preds + targets - preds*targets).sum((2,3))
    return ((inter+eps)/(union+eps)).mean().item()

@torch.no_grad()
def f1_from_probs(probs, targets, eps=1e-7, thr=0.5):
    preds = (probs > thr).float()
    tp = (preds*targets).sum((2,3))
    fp = (preds*(1-targets)).sum((2,3))
    fn = ((1-preds)*targets).sum((2,3))
    prec = (tp+eps)/(tp+fp+eps)
    rec  = (tp+eps)/(tp+fn+eps)
    return (2*prec*rec/(prec+rec+eps)).mean().item()

# =================== EVAL (TTA optional) ===================
def tta_predict(model, x, amp=False, scales=(1.0, 1.25), hflip=True):
    outs = []
    B, C, H, W = x.shape
    for s in scales:
        if s == 1.0:
            xs = x
        else:
            Hs, Ws = int(round(H*s)), int(round(W*s))
            xs = torch.nn.functional.interpolate(x, size=(Hs, Ws), mode="bilinear", align_corners=False)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            lo = model(xs); po = torch.sigmoid(lo)
        if s != 1.0:
            po = torch.nn.functional.interpolate(po, size=(H, W), mode="bilinear", align_corners=False)
        outs.append(po)
        if hflip:
            xh = torch.flip(xs, dims=[-1])
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                lh = model(xh); ph = torch.sigmoid(lh)
            ph = torch.flip(ph, dims=[-1])
            if s != 1.0:
                ph = torch.nn.functional.interpolate(ph, size=(H, W), mode="bilinear", align_corners=False)
            outs.append(ph)
    return torch.stack(outs, dim=0).mean(dim=0)

def run_eval(model, loader, device, amp=False, thr=0.5, use_tta=False):
    model.eval(); total=0; d=i=f=f1=0.0
    with torch.no_grad():
        for x,y,_ in loader:
            x,y = x.to(device), y.to(device)
            if use_tta:
                probs = tta_predict(model, x, amp=amp, scales=TTA_SCALES)
                logits = torch.logit(torch.clamp(probs, 1e-6, 1-1e-6))
            else:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
                    logits = model(x); probs = torch.sigmoid(logits)
            d  += dice_from_logits(logits, y) * x.size(0)
            i  += iou_from_probs(probs, y, thr=thr) * x.size(0)
            f1 += f1_from_probs(probs, y, thr=thr) * x.size(0)
            total += x.size(0)
    return d/total, i/total, f1/total

# =================== TRAIN ===================
def train_main(busi_root: str, runs_dir: str, include_normal=False, img_size=512,
               batch_size=8, epochs=120, lr=1e-4, workers=2, amp=True, seed=42):
    set_seed(seed)

    # build a single base ds for listing files & splitting
    base_list_ds = BUSISegDataset(str(DATA_BASE), image_size=img_size, include_normal=include_normal, train=False)
    n_total = len(base_list_ds)
    n_tr = int(0.8*n_total); n_va = int(0.1*n_total); n_te = n_total - n_tr - n_va
    print(f"Total={n_total} -> train={n_tr}, val={n_va}, test={n_te}")

    g = torch.Generator().manual_seed(seed)
    tr_idx, va_idx, te_idx = random_split(base_list_ds, [n_tr, n_va, n_te], generator=g)

    # Build concrete train/val/test datasets (train has augs; val/test no augs)
    train_full = BUSISegDataset(str(DATA_BASE), image_size=img_size, include_normal=include_normal, train=True)
    val_full   = BUSISegDataset(str(DATA_BASE), image_size=img_size, include_normal=include_normal, train=False)
    test_full  = BUSISegDataset(str(DATA_BASE), image_size=img_size, include_normal=include_normal, train=False)

    train_ds = Subset(train_full, tr_idx.indices)
    val_ds   = Subset(val_full,   va_idx.indices)
    test_ds  = Subset(test_full,  te_idx.indices)

    # tiny-lesion oversampling on the TRAIN subset
    sampler = None
    if train_full.mask_area_frac is not None:
        areas = train_full.mask_area_frac[np.array(tr_idx.indices)]
        thr = np.quantile(areas, TINY_Q)
        w = np.ones_like(areas, dtype=np.float32)
        w[areas <= thr] = TINY_BOOST
        sampler = WeightedRandomSampler(weights=torch.from_numpy(w),
                                        num_samples=len(w), replacement=True)
        print(f"Sampler: tiny<=Q{int(TINY_Q*100)} boosted x{TINY_BOOST:.1f} (thr={thr:.6f}) on train subset")

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=(sampler is None), sampler=sampler,
                              num_workers=workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=workers, pin_memory=True)

    model = build_model().to(device)
    n_params = sum(p.numel() for p in model.parameters())/1e6
    print(f"Params: {n_params:.2f}M")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    criterion = BCE_Dice_FT(0.4, 0.4, 0.2)
    scaler = torch.amp.GradScaler(device="cuda", enabled=(amp and device.type=="cuda"))

    os.makedirs(runs_dir, exist_ok=True)
    best_dice, best_path = -1.0, os.path.join(runs_dir, "best.pt")
    patience, bad_epochs = 15, 0

    for ep in range(1, epochs+1):
        model.train(); run_loss=0.0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs} [train]")
        for x,y,_ in pbar:
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(amp and device.type=="cuda")):
                logits = model(x); loss = criterion(logits, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            # per-iter cosine restarts step
            scheduler.step(ep + (pbar.n / max(1, len(pbar))))
            run_loss += loss.item()*x.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        tr_loss = run_loss/len(train_ds)

        vd, vi, vf = run_eval(model, val_loader, device, amp=amp, thr=0.5, use_tta=False)
        print(f"Epoch {ep}: train_loss={tr_loss:.4f} | val_dice={vd:.4f} | val_iou={vi:.4f} | val_f1={vf:.4f}")

        if vd > best_dice:
            best_dice = vd; bad_epochs = 0
            torch.save({"model": model.state_dict(),
                        "img_size": img_size,
                        "encoder": "timm-efficientnet-b7",
                        "epoch": ep,
                        "val_dice": vd}, best_path)
            print(f"  ✓ Saved new best (Dice={vd:.4f}) → {best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {ep} (no val Dice improvement for {patience} epochs)")
                break

    # Reload best, threshold sweep on VAL (no TTA), then TEST with TTA
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt["model"])

    def sweep_threshold(loader, steps=11, use_tta=False):
        best_thr, best_iou = 0.5, -1.0
        for k in range(steps):
            thr = 0.3 + k*(0.4/(steps-1))  # 0.3..0.7
            d,i,f = run_eval(model, loader, device, amp=amp, thr=thr, use_tta=use_tta)
            if i > best_iou:
                best_iou, best_thr = i, thr
        return best_thr, best_iou

    best_thr, val_iou_at_thr = sweep_threshold(val_loader, steps=11, use_tta=False)
    print(f"Chosen threshold from VAL (no TTA): {best_thr:.3f} (val IoU={val_iou_at_thr:.4f})")

    td, ti, tf1 = run_eval(model, test_loader, device, amp=amp, thr=best_thr, use_tta=True)
    print(f"TEST (with TTA) → Dice={td:.4f} | IoU={ti:.4f} | F1={tf1:.4f}  @thr={best_thr:.3f}")
    print("Best checkpoint:", best_path)

# =================== GO ===================
print("🚀 Training BIG UNet++(EffB7) with heavy dropout and US-friendly augs")
train_main(
    BUSI_BASE, RUNS_DIR,
    include_normal=INCLUDE_NORMAL,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    workers=WORKERS,
    amp=AMP,
    seed=SEED
)


Device: cuda
DATA_DIR is None -> downloading with kagglehub…
kagglehub path: /kaggle/input/breast-ultrasound-images-dataset
Found dataset folder: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Using BUSI_ROOT: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
Resolved data base: /kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT
🚀 Training BIG UNet++(EffB7) with heavy dropout and US-friendly augs
Total=647 -> train=517, val=64, test=66
Sampler: tiny<=Q30 boosted x2.0 (thr=0.028477) on train subset
Params: 74.03M


Epoch 1/120 [train]: 100%|██████████| 130/130 [06:47<00:00,  3.14s/it, loss=0.6529]


Epoch 1: train_loss=0.7657 | val_dice=0.0862 | val_iou=0.5103 | val_f1=0.6179
  ✓ Saved new best (Dice=0.0862) → /content/runs_busi_unetpp_effb7_big/best.pt


Epoch 2/120 [train]: 100%|██████████| 130/130 [00:51<00:00,  2.54it/s, loss=0.6045]


Epoch 2: train_loss=0.5559 | val_dice=0.2907 | val_iou=0.5687 | val_f1=0.6759
  ✓ Saved new best (Dice=0.2907) → /content/runs_busi_unetpp_effb7_big/best.pt


...

Epoch 59: train_loss=0.0652 | val_dice=0.7380 | val_iou=0.6850 | val_f1=0.7555


Epoch 60/120 [train]: 100%|██████████| 130/130 [00:51<00:00,  2.54it/s, loss=0.0548]


Epoch 60: train_loss=0.0646 | val_dice=0.7366 | val_iou=0.6839 | val_f1=0.7540


Epoch 61/120 [train]: 100%|██████████| 130/130 [00:51<00:00,  2.54it/s, loss=0.0366]


Epoch 61: train_loss=0.0647 | val_dice=0.7348 | val_iou=0.6808 | val_f1=0.7520
Early stopping at epoch 61 (no val Dice improvement for 15 epochs)
Chosen threshold from VAL (no TTA): 0.340 (val IoU=0.6907)
TEST (with TTA) → Dice=0.7706 | IoU=0.7427 | F1=0.8267  @thr=0.340
Best checkpoint: /content/runs_busi_unetpp_effb7_big/best.pt
